# Детекция ботов в диалогах. Семантические эмбеддинги и гибридные модели

Постановка задачи: исследование семантических представлений текста на базе компактной языковой модели RuBERT-tiny2 и построение гибридных моделей.

Цели этапа:
- дообучение языковой модели cointegrated/rubert-tiny2 на текстах диалогов
- извлечение эмбеддингов и оценка парной семантической близости собеседников
- сравнение сценариев слияния табличных признаков и эмбеддингов на кросс-валидации
- оптимизация гиперпараметров и обучение гибридных ансамблей CatBoost, LightGBM и XGBoost

In [1]:
import json

import mlflow
import numpy as np
import polars as pl
import xgboost as xgb
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, log_loss, roc_auc_score
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from transformers import AutoModelForSequenceClassification, AutoTokenizer

from project import Reproducibility
from project.core.config import config
from project.infrastructure.clickhouse import FeatureDWH
from project.models import RuBERTBackboneTrainer
from project.models.classical import (
    CatBoostTrainer,
    ClassicalTrainerConfig,
    LightGBMTrainer,
    XGBoostTrainer,
)
from project.tracking import ExperimentTrackerBuilder

2026/11/09 22:34:46 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /home/alex/cu/prod-ml/a.gorobchenko/project/.venv/lib/python3.13/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


In [2]:
Reproducibility.seed(42)

## Дообучение языковой модели RuBERT-tiny2

дообучим архитектуру cointegrated/rubert-tiny2 на последовательностях реплик диалогов с трекингом в MLflow

In [3]:
dwh = FeatureDWH()

train_df = dwh.raw.load_participant_texts("train")

In [4]:
rubert_trainer = RuBERTBackboneTrainer(model_name="cointegrated/rubert-tiny2")

rubert_results = rubert_trainer.fit(
    train_data=train_df,
    max_epochs=100,
    patience=10,
    batch_size=64,
    lr=3e-5,
    weight_decay=0.01,
    max_length=128,
)

/home/alex/cu/prod-ml/a.gorobchenko/project/.venv/lib/python3.13/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


2026/11/09 22:49:53 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


2026/11/09 22:49:53 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in /home/alex/cu/prod-ml/a.gorobchenko/project


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (3 outputs left out)

2026/11/09 22:49:55 INFO mlflow.utils.environment: Successfully exported 230 requirements from uv project. Skipping package capture based inference.


Successfully registered model 'rubert_detector'.
2026/11/09 22:49:57 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: rubert_detector, version 1


Created version '1' of model 'rubert_detector'.


🏃 View run rubert_tiny2_finetuning at: http://127.0.0.1:5000/#/experiments/7/runs/287d2143aa31447a85cbb473f01d2890
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/7


In [5]:
pl.DataFrame(
    {
        "stage": ["RuBERT-tiny2 Fine-Tuning"],
        "best_epoch": [rubert_results["best_epoch"]],
        "val_logloss": [round(rubert_results["final_val_logloss"], 5)],
        "val_roc_auc": [round(rubert_results["final_val_auc"], 5)],
        "val_accuracy": [round(rubert_results["final_val_acc"], 5)],
        "brier_score": [round(rubert_results["final_brier_score"], 5)],
    }
)

stage,best_epoch,val_logloss,val_roc_auc,val_accuracy,brier_score
str,i64,f64,f64,f64,f64
"""RuBERT-tiny2 Fine-Tuning""",10,0.52505,0.80826,0.75248,0.17311


## Загрузка обучающей выборки и семантических эмбеддингов

выгрузим диалоги из ClickHouse и подготовим эмбеддинги RuBERT-tiny2

In [6]:
dwh = FeatureDWH()

train_dialogs = dwh.raw.load_dialogues_dict("train")
test_dialogs = dwh.raw.load_dialogues_dict("test")
ytrain = dwh.raw.load_labels("train")
ytest = dwh.raw.load_labels("test")

y_train = ytrain["is_bot"].to_numpy().astype(int)
groups = ytrain["dialog_id"].to_numpy()

In [7]:
# Извлечение агрегированных текстов участников из ClickHouse DWH
test_df = dwh.raw.load_participant_texts("test")

p_texts_tr = train_df["text"].to_list()
p_texts_te = test_df["text"].to_list()

In [8]:
# Извлечение 312-мерных эмбеддингов в память без сохранения файлов на диск
tok = getattr(rubert_trainer, "tokenizer_", None) or AutoTokenizer.from_pretrained(rubert_trainer.model_name)
bb_model = getattr(rubert_trainer, "model_", None) or AutoModelForSequenceClassification.from_pretrained(
    rubert_trainer.model_name, num_labels=2
).to(rubert_trainer.device)

x_emb_tr = rubert_trainer.extract_embeddings(bb_model, tok, p_texts_tr)
x_emb_te = rubert_trainer.extract_embeddings(bb_model, tok, p_texts_te)

In [9]:
pl.DataFrame(
    {
        "split": ["train", "test"],
        "participants": [len(x_emb_tr), len(x_emb_te)],
        "embedding_dim": [x_emb_tr.shape[1], x_emb_te.shape[1]],
        "bot_ratio": [round(float(np.mean(y_train)), 4), None],
    }
)

split,participants,embedding_dim,bot_ratio
str,i64,i64,f64
"""train""",3032,312,0.408
"""test""",758,312,null


## Попарная семантическая близость собеседников

вычислим косинусную близость между эмбеддингами участников для оценки связности диалога

In [10]:
def compute_pair_cosine_similarity(emb_array: np.ndarray) -> np.ndarray:
    n_samples = emb_array.shape[0]
    cosine_sims = np.zeros(n_samples)
    for i in range(0, n_samples, 2):
        e0 = emb_array[i]
        e1 = emb_array[i + 1]
        norm0 = np.linalg.norm(e0) + 1e-9
        norm1 = np.linalg.norm(e1) + 1e-9
        cos_val = float(np.dot(e0, e1) / (norm0 * norm1))
        cosine_sims[i] = cos_val
        cosine_sims[i + 1] = cos_val
    return cosine_sims


cos_sim_tr = compute_pair_cosine_similarity(x_emb_tr)
cos_sim_te = compute_pair_cosine_similarity(x_emb_te)

pl.DataFrame(
    {
        "metric": ["mean", "std", "min", "median", "max"],
        "train_pair_cos_sim": [
            round(float(np.mean(cos_sim_tr)), 4),
            round(float(np.std(cos_sim_tr)), 4),
            round(float(np.min(cos_sim_tr)), 4),
            round(float(np.median(cos_sim_tr)), 4),
            round(float(np.max(cos_sim_tr)), 4),
        ],
    }
)

metric,train_pair_cos_sim
str,f64
"""mean""",0.1218
"""std""",0.5361
"""min""",-0.7486
"""median""",0.0391
"""max""",1.0


## Семантический классификатор на эмбеддингах

построим OOF-предсказания вероятностей принадлежности к классу бота с помощью LogisticRegression

In [11]:
scaler = StandardScaler()
X_emb_scaled_tr = scaler.fit_transform(x_emb_tr)
X_emb_scaled_te = scaler.transform(x_emb_te)

gkf = GroupKFold(n_splits=5)
oof_rubert = np.zeros(len(y_train))
test_rubert_preds = np.zeros(len(x_emb_te))

for fold, (tr_idx, val_idx) in enumerate(gkf.split(X_emb_scaled_tr, y_train, groups)):
    clf = LogisticRegression(C=0.01, max_iter=1000, random_state=42 + fold)
    clf.fit(X_emb_scaled_tr[tr_idx], y_train[tr_idx])
    oof_rubert[val_idx] = clf.predict_proba(X_emb_scaled_tr[val_idx])[:, 1]
    test_rubert_preds += clf.predict_proba(X_emb_scaled_te)[:, 1] / 5.0

rubert_standalone_loss = log_loss(y_train, oof_rubert)
rubert_standalone_auc = roc_auc_score(y_train, oof_rubert)
rubert_standalone_acc = accuracy_score(y_train, (oof_rubert > 0.5).astype(int))
rubert_standalone_f1 = f1_score(y_train, (oof_rubert > 0.5).astype(int))

pl.DataFrame(
    {
        "model": ["RuBERT-tiny2 (LogReg C=0.01)"],
        "cv_logloss": [round(rubert_standalone_loss, 5)],
        "roc_auc": [round(rubert_standalone_auc, 5)],
        "accuracy": [round(rubert_standalone_acc, 4)],
        "f1_score": [round(rubert_standalone_f1, 4)],
    }
)

model,cv_logloss,roc_auc,accuracy,f1_score
str,f64,f64,f64,f64
"""RuBERT-tiny2 (LogReg C=0.01)""",0.69075,0.52607,0.5656,0.253


## Логирование семантической модели в MLflow

зафиксируем результаты классификатора на эмбеддингах в MLflow

In [12]:
sub_rubert_df = pl.DataFrame({"ID": ytest["ID"].to_numpy(), "is_bot": test_rubert_preds})
emb_feat_names = [f"rubert_emb_{i}" for i in range(X_emb_scaled_tr.shape[1])]

tracker_standalone = (
    ExperimentTrackerBuilder(run_name="rubert_tiny2_semantic_classifier", experiment_name="bot-detection")
    .set_description(
        "Автономный семантический классификатор на базе дообученных эмбеддингов RuBERT-tiny2 и Logistic Regression."
    )
    .set_dataset(
        data=pl.DataFrame({col: X_emb_scaled_tr[:, i] for i, col in enumerate(emb_feat_names)}),
        name="rubert_tiny2_embeddings_312",
        targets=y_train,
    )
    .set_params(
        model_family="RuBERT_Tiny2_LogisticRegression",
        embedding_dim=312,
        regularization="L2",
        C=0.01,
        max_iter=1000,
        solver="lbfgs",
    )
    .require_feature_tree(False)
    .defer_metrics()
    .build()
)

with tracker_standalone:
    tracker_standalone.log_metrics(
        val_logloss=float(rubert_standalone_loss),
        val_roc_auc=float(rubert_standalone_auc),
        val_accuracy=float(rubert_standalone_acc),
        val_f1=float(rubert_standalone_f1),
    )
    tracker_standalone.log_feature_tree(
        {
            "architecture": "LogisticRegression",
            "features": [
                {"name": f, "importance": float(np.abs(clf.coef_[0][i]))} for i, f in enumerate(emb_feat_names)
            ],
        }
    )
    tracker_standalone.log_all_graphics(y_true=y_train, y_pred_proba=oof_rubert)
    tracker_standalone.log_submission(sub_rubert_df, filename="submission.csv")

    rubert_run_id = tracker_standalone.active_run.info.run_id

    dwh.predictions.save_predictions(
        run_id=rubert_run_id,
        run_name="rubert_tiny2_semantic_classifier",
        dataset_split="train_oof",
        dialog_ids=ytrain["dialog_id"].to_numpy(),
        participant_indices=ytrain["participant_index"].to_numpy(),
        prob_bot=oof_rubert,
        prob_bot_calibrated=oof_rubert,
        is_bot=y_train,
    )
    dwh.predictions.save_predictions(
        run_id=rubert_run_id,
        run_name="rubert_tiny2_semantic_classifier",
        dataset_split="test",
        dialog_ids=ytest["dialog_id"].to_numpy(),
        participant_indices=ytest["participant_index"].to_numpy(),
        prob_bot=test_rubert_preds,
        prob_bot_calibrated=test_rubert_preds,
    )

pl.DataFrame(
    {
        "model": ["RuBERT-tiny2 Standalone"],
        "val_logloss": [round(float(rubert_standalone_loss), 5)],
        "val_roc_auc": [round(float(rubert_standalone_auc), 5)],
        "status": ["logged_in_mlflow_and_dwh"],
    }
)

/home/alex/cu/prod-ml/a.gorobchenko/project/.venv/lib/python3.13/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


🏃 View run rubert_tiny2_semantic_classifier at: http://127.0.0.1:5000/#/experiments/7/runs/596371700d8f443d9dcfecc863948805
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/7


model,val_logloss,val_roc_auc,status
str,f64,f64,str
"""RuBERT-tiny2 Standalone""",0.69075,0.52607,"""logged_in_mlflow_and_dwh"""


## Подготовка гибридного признакового пространства

загрузим срез candidate_features из ClickHouse и объединим с отобранными признаками базовых моделей

In [13]:
df_cands_tr, all_feature_names = dwh.features.load_snapshot("candidate_features", dataset_type="train")
df_cands_te, _ = dwh.features.load_snapshot("candidate_features", dataset_type="test")

X_full_tr = df_cands_tr.select(all_feature_names).to_numpy()
X_full_te = df_cands_te.select(all_feature_names).to_numpy()
feat_to_idx = {f: i for i, f in enumerate(all_feature_names)}

pl.DataFrame(
    {
        "snapshot_tag": ["candidate_features"],
        "features_count": [len(all_feature_names)],
        "train_shape": [str(X_full_tr.shape)],
        "test_shape": [str(X_full_te.shape)],
    }
)

snapshot_tag,features_count,train_shape,test_shape
str,i64,str,str
"""candidate_features""",265,"""(3032, 265)""","""(758, 265)"""


In [14]:
# Проверка выравнивания строк среза признаков с целевым вектором ytrain
assert (df_cands_tr["dialog_id"] == ytrain["dialog_id"]).all()
assert (df_cands_tr["participant_index"] == ytrain["participant_index"]).all()
assert (df_cands_te["dialog_id"] == ytest["dialog_id"]).all()
assert (df_cands_te["participant_index"] == ytest["participant_index"]).all()

In [15]:
# Загрузка оптимальных подмножеств признаков из артефактов MLflow (этап 20)
client = mlflow.tracking.MlflowClient(config.mlflow_tracking_uri)
exp = client.get_experiment_by_name("bot-detection")


def load_pruned_feature_names(run_name: str) -> list[str]:
    runs = client.search_runs(
        exp.experiment_id,
        filter_string=f"tags.mlflow.runName = '{run_name}'",
        order_by=["attributes.start_time DESC"],
    )
    if not runs:
        raise ValueError(f"Run '{run_name}' not found in MLflow!")

    art = client.download_artifacts(runs[0].info.run_id, "feature_tree.json")
    with open(art, encoding="utf-8") as f:
        tree = json.load(f)
        return [item["name"] for item in tree["features"]]


cb_top_feats = load_pruned_feature_names("catboost_top_pruned")
lgb_top_feats = load_pruned_feature_names("lightgbm_top_pruned")
xgb_top_feats = load_pruned_feature_names("xgboost_top_pruned")

In [16]:
# Табличные базисы для каждой модели
X_cb_tab_tr = X_full_tr[:, [feat_to_idx[f] for f in cb_top_feats]]
X_cb_tab_te = X_full_te[:, [feat_to_idx[f] for f in cb_top_feats]]
X_lgb_tab_tr = X_full_tr[:, [feat_to_idx[f] for f in lgb_top_feats]]
X_lgb_tab_te = X_full_te[:, [feat_to_idx[f] for f in lgb_top_feats]]
X_xgb_tab_tr = X_full_tr[:, [feat_to_idx[f] for f in xgb_top_feats]]
X_xgb_tab_te = X_full_te[:, [feat_to_idx[f] for f in xgb_top_feats]]

# SVD-15 проекции эмбеддингов RuBERT
svd_rubert = TruncatedSVD(n_components=15, random_state=42)

X_rubert_svd_tr = svd_rubert.fit_transform(X_emb_scaled_tr)
X_rubert_svd_te = svd_rubert.transform(X_emb_scaled_te)

rubert_svd_feature_names = [f"rubert_svd_{i}" for i in range(15)]

In [17]:
# Конфигурации интеграции семантических признаков
semantic_variants = {
    "tabular_only": {
        "description": "только отобранные табличные признаки",
        "tr": np.empty((len(y_train), 0)),
        "te": np.empty((len(ytest), 0)),
        "names": [],
    },
    "svd_embeddings": {
        "description": "табличные + SVD-15 эмбеддингов RuBERT",
        "tr": X_rubert_svd_tr,
        "te": X_rubert_svd_te,
        "names": rubert_svd_feature_names,
    },
    "compact_semantics": {
        "description": "табличные + косинусная близость + OOF-вероятность RuBERT",
        "tr": np.column_stack([cos_sim_tr, oof_rubert]),
        "te": np.column_stack([cos_sim_te, test_rubert_preds]),
        "names": ["semantic_coherence", "rubert_proba"],
    },
    "full_semantics": {
        "description": "табличные + косинусная близость + SVD-15 + OOF-вероятность RuBERT",
        "tr": np.column_stack([cos_sim_tr, X_rubert_svd_tr, oof_rubert]),
        "te": np.column_stack([cos_sim_te, X_rubert_svd_te, test_rubert_preds]),
        "names": ["semantic_coherence"] + rubert_svd_feature_names + ["rubert_proba"],
    },
}

pl.DataFrame(
    {
        "variant": list(semantic_variants.keys()),
        "description": [v["description"] for v in semantic_variants.values()],
        "semantic_features_count": [len(v["names"]) for v in semantic_variants.values()],
    }
)

variant,description,semantic_features_count
str,str,i64
"""tabular_only""","""только отобранные табличные пр…",0
"""svd_embeddings""","""табличные + SVD-15 эмбеддингов…",15
"""compact_semantics""","""табличные + косинусная близост…",2
"""full_semantics""","""табличные + косинусная близост…",17


## Сравнительный анализ конфигураций интеграции семантики

оценим влияние различных вариантов обогащения семантикой на 5 фолдах


In [18]:
dialog_lens = np.array([len(train_dialogs.get(str(d), [])) for d in ytrain["dialog_id"]])
short_mask = dialog_lens <= 2
long_mask = dialog_lens > 2

comparison_records = []
comparison_metrics = {}

for key, var in semantic_variants.items():
    mat = np.column_stack([X_xgb_tab_tr, var["tr"]]) if var["names"] else X_xgb_tab_tr
    oof_m = np.zeros(len(y_train))

    for tr_idx, val_idx in gkf.split(mat, y_train, groups):
        clf = xgb.XGBClassifier(
            n_estimators=300,
            max_depth=5,
            learning_rate=0.05,
            early_stopping_rounds=30,
            random_state=42,
            eval_metric="logloss",
            tree_method="hist",
            n_jobs=4,
        )
        clf.fit(
            mat[tr_idx],
            y_train[tr_idx],
            eval_set=[(mat[val_idx], y_train[val_idx])],
            verbose=False,
        )
        oof_m[val_idx] = clf.predict_proba(mat[val_idx])[:, 1]

    loss_all = float(log_loss(y_train, oof_m))
    loss_s = float(log_loss(y_train[short_mask], oof_m[short_mask]))
    loss_l = float(log_loss(y_train[long_mask], oof_m[long_mask]))
    auc_all = float(roc_auc_score(y_train, oof_m))

    rec = {
        "variant": key,
        "description": var["description"],
        "n_features": mat.shape[1],
        "logloss_overall": round(loss_all, 5),
        "logloss_short": round(loss_s, 5),
        "logloss_long": round(loss_l, 5),
        "roc_auc": round(auc_all, 5),
    }
    comparison_records.append(rec)
    comparison_metrics[key] = rec

df_scenarios = pl.DataFrame(comparison_records).sort("logloss_overall")
df_scenarios

variant,description,n_features,logloss_overall,logloss_short,logloss_long,roc_auc
str,str,i64,f64,f64,f64,f64
"""compact_semantics""","""табличные + косинусная близост…",69,0.17403,0.42566,0.15626,0.9803
"""tabular_only""","""только отобранные табличные пр…",67,0.17472,0.43296,0.15648,0.98027
"""svd_embeddings""","""табличные + SVD-15 эмбеддингов…",82,0.17712,0.42346,0.15972,0.97988
"""full_semantics""","""табличные + косинусная близост…",84,0.17752,0.42672,0.15992,0.97986


In [19]:
# Выбор оптимальной конфигурации семантики среди гибридных вариантов
hybrid_variants = {k: v for k, v in semantic_variants.items() if v["names"]}
best_variant_key = min(hybrid_variants.keys(), key=lambda k: comparison_metrics[k]["logloss_overall"])
best_variant = semantic_variants[best_variant_key]


def assemble_hybrid_data(
    tab_tr: np.ndarray, tab_te: np.ndarray, tab_feat_names: list[str], variant: dict
) -> tuple[np.ndarray, np.ndarray, list[str]]:
    if variant["names"]:
        tr = np.column_stack([tab_tr, variant["tr"]])
        te = np.column_stack([tab_te, variant["te"]])
        names = tab_feat_names + variant["names"]
    else:
        tr, te, names = tab_tr, tab_te, tab_feat_names
    return tr, te, names


X_cb_hyb_tr, X_cb_hyb_te, cb_hyb_feat_names = assemble_hybrid_data(X_cb_tab_tr, X_cb_tab_te, cb_top_feats, best_variant)
X_lgb_hyb_tr, X_lgb_hyb_te, lgb_hyb_feat_names = assemble_hybrid_data(
    X_lgb_tab_tr, X_lgb_tab_te, lgb_top_feats, best_variant
)
X_xgb_hyb_tr, X_xgb_hyb_te, xgb_hyb_feat_names = assemble_hybrid_data(
    X_xgb_tab_tr, X_xgb_tab_te, xgb_top_feats, best_variant
)

pl.DataFrame(
    {
        "best_variant": [best_variant_key],
        "description": [best_variant["description"]],
        "added_semantic_features": [", ".join(best_variant["names"])],
        "catboost_features": [len(cb_hyb_feat_names)],
        "lightgbm_features": [len(lgb_hyb_feat_names)],
        "xgboost_features": [len(xgb_hyb_feat_names)],
    }
)

best_variant,description,added_semantic_features,catboost_features,lightgbm_features,xgboost_features
str,str,str,i64,i64,i64
"""compact_semantics""","""табличные + косинусная близост…","""semantic_coherence, rubert_pro…",61,109,69


## Оптимизация гиперпараметров и обучение CatBoost

подберем гиперпараметры гибридной модели CatBoost в Optuna и обучим ансамбль по фолдам

In [20]:
trainer_cb_hybrid = CatBoostTrainer(
    config=ClassicalTrainerConfig(
        run_name="catboost_rubert_hybrid",
        experiment_name="bot-detection",
        description=f"Гибридный ансамбль CatBoost ({best_variant_key}): {len(cb_hyb_feat_names)} признаков ({len(cb_top_feats)} табл. + {len(best_variant['names'])} семант.: {', '.join(best_variant['names'])}).",
        n_trials=300,
        patience=50,
        dataset_name="train_hybrid_rubert",
    )
)

tuning_cb_hybrid = trainer_cb_hybrid.tune(
    X=X_cb_hyb_tr,
    y=y_train,
    groups=groups,
    feature_names=cb_hyb_feat_names,
)

result_cb_hybrid = trainer_cb_hybrid.fit(
    X_train=X_cb_hyb_tr,
    y_train=y_train,
    groups=groups,
    X_test=X_cb_hyb_te,
    dialog_ids_train=ytrain["dialog_id"].to_numpy(),
    participant_indices_train=ytrain["participant_index"].to_numpy(),
    dialog_ids_test=ytest["dialog_id"].to_numpy(),
    participant_indices_test=ytest["participant_index"].to_numpy(),
    test_ids=ytest["ID"].to_numpy(),
    feature_names=cb_hyb_feat_names,
)

/home/alex/cu/prod-ml/a.gorobchenko/project/.venv/lib/python3.13/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
[I 2026-11-09 22:50:22,479] A new study created in memory with name: no-name-617cc864-977b-441d-9182-31d3a0e879c1


[I 2026-11-09 22:50:32,921] Trial 0 finished with value: 0.16869407260165684 and parameters: {'learning_rate': 0.03268396148966359, 'depth': 7, 'l2_leaf_reg': 7.587945476302646, 'random_strength': 0.24810409748678125}. Best is trial 0 with value: 0.16869407260165684.


[I 2026-11-09 22:50:37,169] Trial 1 finished with value: 0.17783351680898987 and parameters: {'learning_rate': 0.020748663673269163, 'depth': 4, 'l2_leaf_reg': 1.5227525095137953, 'random_strength': 2.9154431891537547}. Best is trial 0 with value: 0.16869407260165684.


[I 2026-11-09 22:50:41,342] Trial 2 finished with value: 0.16620427008609825 and parameters: {'learning_rate': 0.052354281909450046, 'depth': 6, 'l2_leaf_reg': 1.185260448662222, 'random_strength': 7.579479953348009}. Best is trial 2 with value: 0.16620427008609825.


[I 2026-11-09 22:50:43,023] Trial 3 finished with value: 0.17430106990747857 and parameters: {'learning_rate': 0.08469579964398871, 'depth': 4, 'l2_leaf_reg': 2.636424704863906, 'random_strength': 0.00541524411940254}. Best is trial 2 with value: 0.16620427008609825.


[I 2026-11-09 22:50:50,408] Trial 4 finished with value: 0.17034764893865426 and parameters: {'learning_rate': 0.02823900447907086, 'depth': 6, 'l2_leaf_reg': 4.887505167779041, 'random_strength': 0.014618962793704957}. Best is trial 2 with value: 0.16620427008609825.


... (194 outputs left out)

[I 2026-11-09 23:20:54,954] Trial 199 finished with value: 0.16762173899840382 and parameters: {'learning_rate': 0.05267335284593097, 'depth': 6, 'l2_leaf_reg': 4.2245301339844215, 'random_strength': 5.551080119133937}. Best is trial 151 with value: 0.1620953725253594.


[I 2026-11-09 23:21:03,296] Trial 200 finished with value: 0.1643082660676502 and parameters: {'learning_rate': 0.04048107118563792, 'depth': 6, 'l2_leaf_reg': 4.759732257885439, 'random_strength': 4.8952495123322235}. Best is trial 151 with value: 0.1620953725253594.


[I 2026-11-09 23:21:10,986] Trial 201 finished with value: 0.16589280730246206 and parameters: {'learning_rate': 0.035463929812315886, 'depth': 6, 'l2_leaf_reg': 2.706183744526492, 'random_strength': 7.085224574404045}. Best is trial 151 with value: 0.1620953725253594.


🏃 View run catboost_rubert_hybrid at: http://127.0.0.1:5000/#/experiments/7/runs/969238860d3d4d65a55d90e4b4bf4a23
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/7


## Оптимизация гиперпараметров и обучение LightGBM

подберем гиперпараметры гибридной модели LightGBM в Optuna и обучим ансамбль по фолдам

In [21]:
trainer_lgb_hybrid = LightGBMTrainer(
    config=ClassicalTrainerConfig(
        run_name="lightgbm_rubert_hybrid",
        experiment_name="bot-detection",
        description=f"Гибридный ансамбль LightGBM ({best_variant_key}): {len(lgb_hyb_feat_names)} признаков ({len(lgb_top_feats)} табл. + {len(best_variant['names'])} семант.: {', '.join(best_variant['names'])}).",
        n_trials=300,
        patience=50,
        dataset_name="train_hybrid_rubert",
    )
)

tuning_lgb_hybrid = trainer_lgb_hybrid.tune(
    X=X_lgb_hyb_tr,
    y=y_train,
    groups=groups,
    feature_names=lgb_hyb_feat_names,
)

result_lgb_hybrid = trainer_lgb_hybrid.fit(
    X_train=X_lgb_hyb_tr,
    y_train=y_train,
    groups=groups,
    X_test=X_lgb_hyb_te,
    dialog_ids_train=ytrain["dialog_id"].to_numpy(),
    participant_indices_train=ytrain["participant_index"].to_numpy(),
    dialog_ids_test=ytest["dialog_id"].to_numpy(),
    participant_indices_test=ytest["participant_index"].to_numpy(),
    test_ids=ytest["ID"].to_numpy(),
    feature_names=lgb_hyb_feat_names,
)

/home/alex/cu/prod-ml/a.gorobchenko/project/.venv/lib/python3.13/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
[I 2026-11-09 23:21:28,064] A new study created in memory with name: no-name-79d4a8bc-d44c-4cc9-9646-87ce5e57a55d


[I 2026-11-09 23:21:30,102] Trial 0 finished with value: 0.17422773648744908 and parameters: {'learning_rate': 0.03268396148966359, 'num_leaves': 61, 'max_depth': 7, 'min_child_samples': 35, 'subsample': 0.6624074561769746, 'colsample_bytree': 0.49359671220172163, 'reg_alpha': 0.0016400214911202306, 'reg_lambda': 1.599409170128044}. Best is trial 0 with value: 0.17422773648744908.


[I 2026-11-09 23:21:31,273] Trial 1 finished with value: 0.17783235611096856 and parameters: {'learning_rate': 0.052354281909450046, 'num_leaves': 49, 'max_depth': 3, 'min_child_samples': 50, 'subsample': 0.9329770563201687, 'colsample_bytree': 0.5274034664069657, 'reg_alpha': 0.004705059281907645, 'reg_lambda': 0.004768785415482609}. Best is trial 0 with value: 0.17422773648744908.


[I 2026-11-09 23:21:33,186] Trial 2 finished with value: 0.17064397946595458 and parameters: {'learning_rate': 0.02823900447907086, 'num_leaves': 40, 'max_depth': 5, 'min_child_samples': 20, 'subsample': 0.8447411578889518, 'colsample_bytree': 0.4836963163912251, 'reg_alpha': 0.012040216379191714, 'reg_lambda': 0.022654864504851786}. Best is trial 2 with value: 0.17064397946595458.


[I 2026-11-09 23:21:34,636] Trial 3 finished with value: 0.17228578805669054 and parameters: {'learning_rate': 0.038722509516311295, 'num_leaves': 53, 'max_depth': 4, 'min_child_samples': 30, 'subsample': 0.836965827544817, 'colsample_bytree': 0.42787024763199866, 'reg_alpha': 0.1767218232266506, 'reg_lambda': 0.0042733023193754}. Best is trial 2 with value: 0.17064397946595458.


[I 2026-11-09 23:21:37,823] Trial 4 finished with value: 0.17120069755269346 and parameters: {'learning_rate': 0.017172704676853088, 'num_leaves': 61, 'max_depth': 8, 'min_child_samples': 45, 'subsample': 0.7218455076693483, 'colsample_bytree': 0.45860326840383037, 'reg_alpha': 0.33959199243313637, 'reg_lambda': 0.042472797953697176}. Best is trial 2 with value: 0.17064397946595458.


... (52 outputs left out)

[I 2026-11-09 23:24:08,354] Trial 57 finished with value: 0.17022480171402957 and parameters: {'learning_rate': 0.028502948987088616, 'num_leaves': 49, 'max_depth': 8, 'min_child_samples': 45, 'subsample': 0.9951451380218143, 'colsample_bytree': 0.42971584618663755, 'reg_alpha': 0.021728497842496065, 'reg_lambda': 0.0019716283616987404}. Best is trial 9 with value: 0.16856410865796195.


[I 2026-11-09 23:24:11,801] Trial 58 finished with value: 0.16934269471574415 and parameters: {'learning_rate': 0.01723592423119279, 'num_leaves': 60, 'max_depth': 8, 'min_child_samples': 45, 'subsample': 0.9417887211852267, 'colsample_bytree': 0.4291034021303325, 'reg_alpha': 0.04757714860911528, 'reg_lambda': 0.002863078403702077}. Best is trial 9 with value: 0.16856410865796195.


[I 2026-11-09 23:24:14,557] Trial 59 finished with value: 0.1706759718213274 and parameters: {'learning_rate': 0.02196856055714695, 'num_leaves': 58, 'max_depth': 8, 'min_child_samples': 45, 'subsample': 0.9847614009843242, 'colsample_bytree': 0.44860788265851737, 'reg_alpha': 0.027497836076954094, 'reg_lambda': 0.0025346739425196364}. Best is trial 9 with value: 0.16856410865796195.


🏃 View run lightgbm_rubert_hybrid at: http://127.0.0.1:5000/#/experiments/7/runs/e0a9447fa0914da79a4332ac8e2b7646
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/7


## Оптимизация гиперпараметров и обучение XGBoost

подберем гиперпараметры гибридной модели XGBoost в Optuna и обучим ансамбль по фолдам

In [22]:
trainer_xgb_hybrid = XGBoostTrainer(
    config=ClassicalTrainerConfig(
        run_name="xgboost_rubert_hybrid",
        experiment_name="bot-detection",
        description=f"Гибридный ансамбль XGBoost ({best_variant_key}): {len(xgb_hyb_feat_names)} признаков ({len(xgb_top_feats)} табл. + {len(best_variant['names'])} семант.: {', '.join(best_variant['names'])}).",
        n_trials=300,
        patience=50,
        dataset_name="train_hybrid_rubert",
    )
)

tuning_xgb_hybrid = trainer_xgb_hybrid.tune(
    X=X_xgb_hyb_tr,
    y=y_train,
    groups=groups,
    feature_names=xgb_hyb_feat_names,
)

result_xgb_hybrid = trainer_xgb_hybrid.fit(
    X_train=X_xgb_hyb_tr,
    y_train=y_train,
    groups=groups,
    X_test=X_xgb_hyb_te,
    dialog_ids_train=ytrain["dialog_id"].to_numpy(),
    participant_indices_train=ytrain["participant_index"].to_numpy(),
    dialog_ids_test=ytest["dialog_id"].to_numpy(),
    participant_indices_test=ytest["participant_index"].to_numpy(),
    test_ids=ytest["ID"].to_numpy(),
    feature_names=xgb_hyb_feat_names,
)

/home/alex/cu/prod-ml/a.gorobchenko/project/.venv/lib/python3.13/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
[I 2026-11-09 23:24:19,390] A new study created in memory with name: no-name-d95e8477-2701-487f-8e06-a1a52fb23bc8


[I 2026-11-09 23:24:20,575] Trial 0 finished with value: 0.17243684232234954 and parameters: {'max_depth': 4, 'learning_rate': 0.10831081647339098, 'subsample': 0.892797576724562, 'colsample_bytree': 0.759195090518222, 'reg_alpha': 0.003776663327107336, 'reg_lambda': 0.003775887545682684}. Best is trial 0 with value: 0.17243684232234954.


[I 2026-11-09 23:24:22,063] Trial 1 finished with value: 0.17844279408454894 and parameters: {'max_depth': 3, 'learning_rate': 0.09085027856438452, 'subsample': 0.8404460046972835, 'colsample_bytree': 0.8248435466776274, 'reg_alpha': 0.0011916299962955155, 'reg_lambda': 3.8696122574142664}. Best is trial 0 with value: 0.17243684232234954.


[I 2026-11-09 23:24:25,306] Trial 2 finished with value: 0.16743204295635222 and parameters: {'max_depth': 7, 'learning_rate': 0.023326661313163164, 'subsample': 0.6727299868828402, 'colsample_bytree': 0.5100427059120604, 'reg_alpha': 0.01334697757417809, 'reg_lambda': 0.08730885649333642}. Best is trial 2 with value: 0.16743204295635222.


[I 2026-11-09 23:24:27,416] Trial 3 finished with value: 0.16722788512706757 and parameters: {'max_depth': 5, 'learning_rate': 0.027485103675404694, 'subsample': 0.8447411578889518, 'colsample_bytree': 0.4836963163912251, 'reg_alpha': 0.012040216379191714, 'reg_lambda': 0.022654864504851786}. Best is trial 3 with value: 0.16722788512706757.


[I 2026-11-09 23:24:28,613] Trial 4 finished with value: 0.17240231931209565 and parameters: {'max_depth': 5, 'learning_rate': 0.07676721897355432, 'subsample': 0.6798695128633439, 'colsample_bytree': 0.708540663048167, 'reg_alpha': 0.15535445807588452, 'reg_lambda': 0.0014853153980421612}. Best is trial 3 with value: 0.16722788512706757.


... (100 outputs left out)

[I 2026-11-09 23:29:59,601] Trial 105 finished with value: 0.1665915846824646 and parameters: {'max_depth': 6, 'learning_rate': 0.016460944503819554, 'subsample': 0.8422169307126998, 'colsample_bytree': 0.4513121124989158, 'reg_alpha': 0.021089675233412968, 'reg_lambda': 0.006439359684924639}. Best is trial 57 with value: 0.16394023597240448.


[I 2026-11-09 23:30:02,211] Trial 106 finished with value: 0.16558530032634736 and parameters: {'max_depth': 6, 'learning_rate': 0.020221519293354737, 'subsample': 0.8362679646217034, 'colsample_bytree': 0.42353039348742616, 'reg_alpha': 0.060749702887011515, 'reg_lambda': 0.008615815592415442}. Best is trial 57 with value: 0.16394023597240448.


[I 2026-11-09 23:30:05,361] Trial 107 finished with value: 0.1660817950963974 and parameters: {'max_depth': 6, 'learning_rate': 0.016391492051772066, 'subsample': 0.8647779744559203, 'colsample_bytree': 0.42634080432506793, 'reg_alpha': 0.12239423586677971, 'reg_lambda': 0.009640912637511457}. Best is trial 57 with value: 0.16394023597240448.


🏃 View run xgboost_rubert_hybrid at: http://127.0.0.1:5000/#/experiments/7/runs/59c90015d85f44efaa55ed99119b68f7
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/7


## Сводная сравнительная таблица гибридных моделей

извлечем метрики кросс-валидации гибридных моделей из MLflow и сопоставим с бейзлайнами

In [23]:
client = mlflow.tracking.MlflowClient(config.mlflow_tracking_uri)
exp = client.get_experiment_by_name("bot-detection")
runs = client.search_runs(exp.experiment_id, order_by=["attributes.start_time DESC"]) if exp else []

hyb_names = [
    "catboost_rubert_hybrid",
    "lightgbm_rubert_hybrid",
    "xgboost_rubert_hybrid",
    "rubert_tiny2_semantic_classifier",
]
rows = []

for rn in hyb_names:
    matched = [r for r in runs if r.data.tags.get("mlflow.runName") == rn]
    if matched:
        m = matched[0].data.metrics
        rows.append(
            {
                "model": rn,
                "val_logloss": round(m.get("val_logloss", 0.0), 5),
                "val_roc_auc": round(m.get("val_roc_auc", 0.0), 5),
                "val_accuracy": round(m.get("val_accuracy", 0.0), 5),
            }
        )

# Сохранение OOF и тестовых предсказаний всех гибридных моделей в ClickHouse DWH
for name, res in [
    ("catboost_rubert_hybrid", result_cb_hybrid),
    ("lightgbm_rubert_hybrid", result_lgb_hybrid),
    ("xgboost_rubert_hybrid", result_xgb_hybrid),
]:
    dwh.predictions.save_predictions(
        run_id=res.run_id,
        run_name=name,
        dataset_split="train_oof",
        dialog_ids=ytrain["dialog_id"].to_numpy(),
        participant_indices=ytrain["participant_index"].to_numpy(),
        prob_bot=res.oof_raw,
        prob_bot_calibrated=res.oof_calibrated,
        is_bot=y_train,
    )
    dwh.predictions.save_predictions(
        run_id=res.run_id,
        run_name=name,
        dataset_split="test",
        dialog_ids=ytest["dialog_id"].to_numpy(),
        participant_indices=ytest["participant_index"].to_numpy(),
        prob_bot=res.test_raw,
        prob_bot_calibrated=res.test_calibrated,
    )

pl.DataFrame(rows).sort("val_logloss")

model,val_logloss,val_roc_auc,val_accuracy
str,f64,f64,f64
"""catboost_rubert_hybrid""",0.16023,0.98372,0.93931
"""xgboost_rubert_hybrid""",0.16309,0.98303,0.93602
"""lightgbm_rubert_hybrid""",0.16481,0.98272,0.93635
"""rubert_tiny2_semantic_classifi…",0.69075,0.52607,0.56563
